In [0]:
%sql
-- Creating schemas in Hive metastore
CREATE DATABASE IF NOT EXISTS bronze;
CREATE DATABASE IF NOT EXISTS silver;
CREATE DATABASE IF NOT EXISTS gold;
CREATE DATABASE IF NOT EXISTS dq;

-- DQ results log 
CREATE TABLE IF NOT EXISTS dq.check_results (
  run_time_stamp TIMESTAMP,
  check_name STRING,
  status STRING,
  failed_count BIGINT,
  details STRING
)
USING DELTA;

Minimal Bronze DQ checks(log to dq_results)

In [0]:
from pyspark.sql import functions as F

bronze_data = spark.table("bronze.appointments_raw")

def log_check(name, failed_count, details=""):
    status = "PASS" if failed_count == 0 else "FAIL"
    (spark.createDataFrame([(name, status, int(failed_count), details)],
                           "check_name STRING, status STRING, failed_count BIGINT, details STRING")
          .withColumn("run_time_stamp", F.current_timestamp())
          .select("run_time_stamp","check_name","status","failed_count","details")
          .write.mode("append").format("delta").saveAsTable("dq.check_results"))

log_check("bronze_rowcount_gt_0", bronze_data.limit(1).count() == 0, "table should not be empty")

log_check("bronze_no_null_patient_or_appt",
          bronze_data.filter(F.col("PatientId").isNull() | F.col("AppointmentID").isNull()).count(),
          "PatientId and AppointmentID must not be null")


In [0]:
%sql
SELECT COUNT(*) FROM bronze.appointments_raw;


In [0]:
%sql
SELECT * FROM dq.check_results LIMIT 5;
